In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/dev/GeoguessrAI/stitchedWithEU.zip'
extract_path = '/content/unzipped'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Unzipped to:", extract_path)


✅ Unzipped to: /content/unzipped


In [ ]:
import timm
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm import tqdm
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score

img_width = 224
img_height = 224
batch_size = 32
num_epochs = 50
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data transforms
transform = transforms.Compose([
    transforms.Resize((img_width, img_height)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

#All country labels are folders with hold all images of that class
dataset = datasets.ImageFolder("unzipped/stitched", transform=transform)

# Train/val split
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

num_classes = len(dataset.classes)

#Load backbone
backbone = timm.create_model('tiny_vit_21m_224', pretrained=True, num_classes=0)

# Custom head on backbone
class CustomTinyViT(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        in_features = backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x

model = CustomTinyViT(backbone, num_classes).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.03)

# Scheduler (with warmup + cosine)
import math
from torch.optim.lr_scheduler import LambdaLR
def cosine_schedule_with_warmup(optimizer, warmup_epochs, total_epochs, final_lr_factor=0.01):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        else:
            progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
            return final_lr_factor + 0.5 * (1 - final_lr_factor) * (1 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda=lr_lambda)

scheduler = cosine_schedule_with_warmup(optimizer, warmup_epochs=5, total_epochs=num_epochs, final_lr_factor=1e-6/3e-4)

# TorchMetrics
accuracy = MulticlassAccuracy(num_classes=num_classes, average="macro").to(device)
precision = MulticlassPrecision(num_classes=num_classes, average="macro").to(device)
recall = MulticlassRecall(num_classes=num_classes, average="macro").to(device)
f1 = MulticlassF1Score(num_classes=num_classes, average="macro").to(device)

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss, total_correct = 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = total_loss / len(train_dataset)
    train_acc = total_correct / len(train_dataset)
    print(f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f}")

    # Validation
    model.eval()
    val_loss, val_correct = 0, 0
    accuracy.reset(), precision.reset(), recall.reset(), f1.reset()

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            preds = outputs.argmax(1)
            val_loss += loss.item() * images.size(0)
            val_correct += (preds == labels).sum().item()

            # update torchmetrics
            accuracy.update(preds, labels)
            precision.update(preds, labels)
            recall.update(preds, labels)
            f1.update(preds, labels)

    val_loss /= len(val_dataset)
    val_acc = val_correct / len(val_dataset)
    val_precision = precision.compute().item()
    val_recall = recall.compute().item()
    val_f1 = f1.compute().item()

    print(f"Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | "
          f"Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | F1: {val_f1:.4f}")

    scheduler.step()


model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Epoch 1/50: 100%|██████████| 7790/7790 [30:33<00:00,  4.25it/s]

Train Loss: 2.5946 Acc: 0.3054


Val Loss: 1.4730 | Acc: 0.5600 | Precision: 0.4641 | Recall: 0.4065 | F1: 0.3995


Epoch 2/50: 100%|██████████| 7790/7790 [30:34<00:00,  4.25it/s]

...

Epoch 27/50: 100%|██████████| 7790/7790 [30:32<00:00,  4.25it/s]

Train Loss: 0.1784 Acc: 0.9453


Val Loss: 0.6058 | Acc: 0.8604 | Precision: 0.8396 | Recall: 0.8423 | F1: 0.8363


Epoch 28/50:  38%|███▊      | 2942/7790 [11:32<19:01,  4.25it/s]


KeyboardInterrupt: 

In [ ]:
#Save model weights
torch.save(model.state_dict(), 'tiny_country.pth')
